# Lisa MB registration (2026_07_30) — condensed

**Per scene:** [2] select → [3] click landmarks (napari) → [4] save → [5] register + warp (no trim yet) →
[6] build / inspect signal envelope, tune params → [7] apply trim + save.
**Re-run an already-clicked scene:** [2] then [5] → [6] → [7] (skip [3]/[4]) — keep `ROT180` as it was clicked.
**Upside-down brain:** set `ROT180 = True` in [2]; masks are saved back in the raw orientation.

Saves per scene, in `series_<s>/masks/`: `msk_MB_registered_s<s>.tif` (trimmed MB mask) and
`msk_MB_registered_s<s>_ALL.tif` (the signal envelope used). Landmarks: `series_<s>/landmarks_s<s>.pkl.gz`.

Run [1] SETUP once. If a cell errors on a template/label shape mismatch or an out-of-bounds
crop, the kernel has a stale cropped atlas from another notebook — **Restart Kernel → run [1] again**.

In [ ]:
# ═══════════════════════════  SETUP  (run once)  ═══════════════════════════
%load_ext autoreload
%autoreload 2
import os, sys, platform, pickle, gzip, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ants
from aicsimageio import AICSImage
from scipy.ndimage import gaussian_filter
from tifffile import imwrite

system = platform.system()
home = {'Linux': '/home/gerard/', 'Darwin': '/Users/gerard/', 'Windows': 'C:/Users/cviko/'}[system]
try:
    sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
    from data_processing import describe_acquisition
except ImportError:
    sys.path.append(home + 'analysis/confocal/src')
    from data_processing import describe_acquisition

data_home = home + 'data/confocal/'
atlas_home = data_home + 'atlas_JFRC2/'
date, user = '2026_07_30', 'Lisa'
lif_path = data_home + date + '_' + user + '/Project.lif'

# --- atlas download (no-op if already present) ---
_ATLAS = {
    'JFRCtemplate2010.nrrd':
        'https://raw.githubusercontent.com/VirtualFlyBrain/DrosAdultBRAINdomains/master/template/JFRCtemplate2010.nrrd',
    'JFRCtempate2010.mask130819_Original.nrrd':
        'https://raw.githubusercontent.com/VirtualFlyBrain/DrosAdultBRAINdomains/master/combinedIndexFiles/JFRCtempate2010.mask130819_Original.nrrd',
    'Original_Index.tsv':
        'https://raw.githubusercontent.com/VirtualFlyBrain/DrosAdultBRAINdomains/master/refData/Original_Index.tsv',
}
os.makedirs(atlas_home, exist_ok=True)
for fn, url in _ATLAS.items():
    if not os.path.exists(atlas_home + fn):
        print('downloading', fn); urllib.request.urlretrieve(url, atlas_home + fn)

# --- atlas + acquisition info ---
template_full = ants.image_read(atlas_home + 'JFRCtemplate2010.nrrd')
label_img_full = ants.image_read(atlas_home + 'JFRCtempate2010.mask130819_Original.nrrd')
MB_LABEL_IDS = [17, 18, 19, 64, 65, 66]   # pedunculus + vertical + medial lobe, R then L (no calyx: 36, 81)
info = describe_acquisition(lif_path, do_print=False)
scene_names = list(info.keys())
print(f'{len(scene_names)} scenes:')
for _i, _s in enumerate(scene_names):
    print(f'  {_i:3d}  {_s}')


# ═══════════════════════════  HELPERS  ═══════════════════════════
def _fit_similarity_allow_reflection(moving_pts, fixed_pts):
    """Umeyama similarity fit (rotation-OR-reflection + uniform scale + translation), without
    ants' forced det=+1 correction -- the template<->scene correspondence here is a genuine
    mirror flip (det=-1); forcing a proper rotation distorts one axis instead."""
    cf, cm = fixed_pts.mean(0), moving_pts.mean(0)
    x, y = fixed_pts - cf, moving_pts - cm
    U, D, Vt = np.linalg.svd(y.T @ x)
    R = U @ Vt
    A = (D.sum() / (x ** 2).sum()) * R
    return A, cm - A @ cf


def _crop_template_to_scene(template_full, label_img_full, nc82_stack, vxy, label_ids, Y_ANCHOR='top'):
    """Crop template + labels to this scene's physical FOV (centered in X, top-anchored in Y);
    build the binary MB template mask. Indices are clamped to the image, so a too-large FOV
    just yields the whole axis instead of an out-of-bounds ITK error."""
    if tuple(template_full.shape) != tuple(label_img_full.shape):
        raise ValueError(f'template {template_full.shape} vs label {label_img_full.shape} shape '
                         f'mismatch -- re-run the SETUP cell (stale cropped image in the kernel).')
    sx, sy, sz = (int(v) for v in template_full.shape)
    cx = min(int(round(nc82_stack.shape[2] * vxy / template_full.spacing[0])), sx)
    cy = min(int(round(nc82_stack.shape[1] * vxy / template_full.spacing[1])), sy)
    x_min = int(np.clip(round(sx / 2 - cx / 2), 0, sx - cx))
    y_min = 0 if Y_ANCHOR == 'top' else sy - cy
    lo, hi = (x_min, y_min, 0), (x_min + cx, y_min + cy, sz)
    print(f'  crop {template_full.shape} -> lo{lo} hi{hi}  '
          f'(scene FOV {nc82_stack.shape[2]*vxy:.0f}x{nc82_stack.shape[1]*vxy:.0f} um)')
    template = ants.crop_indices(template_full, lo, hi)
    label_img = ants.crop_indices(label_img_full, lo, hi)
    mb = np.isin(label_img.numpy(), label_ids).astype(np.float32)
    mb_tpl = ants.from_numpy(mb, origin=label_img.origin, spacing=label_img.spacing,
                             direction=label_img.direction)
    return template, mb_tpl


# --- signal envelope, split so the expensive part (the blur) is done once and reused ---
def _blurred_stack(stack, vxy, vz, sigma_um=0.8):
    return gaussian_filter(stack.astype(np.float32),
                           sigma=[sigma_um / vz, sigma_um / vxy, sigma_um / vxy])


def get_blurred(P, sigma_um=0.8):
    """Blur the scene's nc82 stack once and cache it on P (keyed by sigma). Returns
    (blurred, frame_mean). Re-tuning the percentiles reuses this -- only changing sigma re-blurs."""
    key = f'_blurred@{sigma_um}'
    if P.get(key) is None:
        P[key] = _blurred_stack(P['nc82_stack'], P['vxy'], P['vz'], sigma_um)
        P['_frame_mean'] = P['nc82_stack'].astype(np.float32).mean(axis=(1, 2))
    return P[key], P['_frame_mean']


def signal_mask_fixed(blurred, percentile=70):
    return blurred > np.percentile(blurred, percentile)


def signal_mask_adaptive(blurred, frame_mean, base_percentile=70, step_percentile=5,
                         step_pct_drop=10, min_percentile=0):
    """Per-frame percentile threshold that steps down for dim frames past the intensity peak.
    All distinct percentile levels are evaluated in ONE np.percentile call (the slow part
    otherwise -- one full pass per frame)."""
    peak = int(np.argmax(frame_mean))
    pcts = np.full(blurred.shape[0], float(base_percentile))
    post = np.arange(blurred.shape[0]) > peak
    nstep = (np.clip((1 - frame_mean / frame_mean[peak]) * 100, 0, None) // step_pct_drop).astype(int)
    pcts[post] = np.maximum(base_percentile - step_percentile * nstep[post], min_percentile)
    levels = np.unique(pcts)
    thr = dict(zip(levels, np.atleast_1d(np.percentile(blurred, levels))))
    out = np.empty(blurred.shape, bool)
    for z in range(blurred.shape[0]):
        out[z] = blurred[z] > thr[pcts[z]]
    return out


def defining_whole_mask(stack, vxy, vz, percentile=70, sigma_um=0.8):
    return signal_mask_fixed(_blurred_stack(stack, vxy, vz, sigma_um), percentile).astype(np.uint8)


def defining_whole_mask_adaptative(stack, vxy, vz, base_percentile=70, step_percentile=5,
                                   step_pct_drop=10, min_percentile=0, sigma_um=0.8):
    b = _blurred_stack(stack, vxy, vz, sigma_um)
    fm = stack.astype(np.float32).mean(axis=(1, 2))
    return signal_mask_adaptive(b, fm, base_percentile, step_percentile, step_pct_drop,
                                min_percentile).astype(np.uint8)


_TUNE_VIEWER = {}


def _viewer_visible(v):
    """True only if the napari window is actually open on screen (a closed Viewer object
    still answers .layers, so that alone isn't enough)."""
    try:
        return v is not None and v.window._qt_window.isVisible()
    except Exception:
        return False


def tune_viewer(scene, P, mb_reg, mask_all, mask_all_adapt):
    """One napari window for envelope tuning. If it's still open, swap the 3 mask layers and
    raise it; if it was closed (or first run), open a fresh one. Never calls .close() on an
    already-destroyed viewer (that throws deep in vispy)."""
    import napari
    names = ('MB (untrimmed)', 'env fixed', 'env adaptive')
    v = _TUNE_VIEWER.get('v')
    if _viewer_visible(v):
        try:
            if all(n in v.layers for n in names):
                v.layers['MB (untrimmed)'].data = mb_reg
                v.layers['env fixed'].data = mask_all
                v.layers['env adaptive'].data = mask_all_adapt
                try:
                    v.window._qt_window.raise_()
                    v.window._qt_window.activateWindow()
                except Exception:
                    pass
                return v
            v.close()                     # visible but unexpected layers -> replace it
        except Exception:
            pass
    _TUNE_VIEWER.pop('v', None)            # stale/closed ref: just drop it, don't touch it
    v = napari.Viewer(title=f'trim tuning — s{scene} ({scene_names[scene]})')
    v.add_image(P['nc82_stack'], name='nc82', colormap='gray')
    v.add_image(mb_reg,         name='MB (untrimmed)', colormap='blue',  opacity=.40)
    v.add_image(mask_all,       name='env fixed',      colormap='red',   opacity=.25, visible=False)
    v.add_image(mask_all_adapt, name='env adaptive',   colormap='green', opacity=.25)
    _TUNE_VIEWER['v'] = v
    return v


def prep_scene(scene, rot180=False):
    """rot180=True: brain imaged 180° from the template's top-down orientation -> rotate the
    nc82 stack in-plane (flip Y and X; shape unchanged) so landmark clicking and registration
    work in a matching frame. Everything downstream stays in this rotated frame; save_mask
    rotates the saved masks back to the raw-stack orientation."""
    vxy = info[scene_names[scene]]['voxel_xy_um']
    vz = info[scene_names[scene]]['voxel_z_um']
    img = AICSImage(lif_path)
    img.set_scene(img.scenes[scene])
    nc82_stack = img.get_image_data('ZYX', T=0, C=0).astype(np.float32)
    if rot180:
        nc82_stack = np.rot90(nc82_stack, 2, axes=(1, 2)).copy()
    moving = ants.from_numpy(nc82_stack, spacing=(vz, vxy, vxy))
    template, mb_mask_template = _crop_template_to_scene(template_full, label_img_full, nc82_stack, vxy, MB_LABEL_IDS)
    return dict(scene=scene, rot180=bool(rot180), vxy=vxy, vz=vz, nc82_stack=nc82_stack,
               moving=moving, template=template, mb_mask_template=mb_mask_template)


def open_landmark_viewers(P):
    """Two napari viewers (template transposed to Z,Y,X to match the scene). Click matching
    landmarks in the SAME ORDER in both, spread across X/Y/Z, then run the save cell."""
    import napari
    tz = np.transpose(P['template'].numpy(), (2, 1, 0))
    mz = np.transpose(P['mb_mask_template'].numpy(), (2, 1, 0))
    vt = napari.Viewer(title='TEMPLATE — click landmarks')
    vt.add_image(tz, name='template', colormap='gray')
    vt.add_image(mz, name='mb_mask', colormap='blue', opacity=0.2)
    tl = vt.add_points(name='landmarks', ndim=3, size=8, face_color='red')
    vs = napari.Viewer(title=f"SCENE {P['scene']} ({scene_names[P['scene']]}) — click landmarks, SAME ORDER")
    vs.add_image(P['nc82_stack'], name='nc82', colormap='gray')
    sl = vs.add_points(name='landmarks', ndim=3, size=8, face_color='red')
    return vt, vs, tl, sl


def _check_points_in_bounds(P, tpl_data, scn_data):
    """Clicked landmark indices must lie inside the images. Out-of-range coords (esp. a scene
    X far above the stack width) mean clicks landed in the wrong viewer or on an offset layer
    -- the similarity fit will then look fine (small residual, sane scale) but place the scene
    mostly outside the template, so the warped nc82 shows 'just a sliver of brain'."""
    checks = [('template', np.asarray(tpl_data), P['template'].shape[::-1]),   # display order (Z,Y,X)
              ('scene', np.asarray(scn_data), tuple(P['moving'].shape))]        # (Z,Y,X)
    for name, d, shape in checks:
        oob = ((d < 0) | (d >= np.array(shape))).any(axis=1)
        if oob.any():
            raise ValueError(
                f'{int(oob.sum())} {name} landmark(s) outside the image {shape} (Z,Y,X): '
                f'rows {list(np.where(oob)[0])} = {d[oob].round(1).tolist()}. '
                f'Re-click that viewer (wrong viewer / offset layer / stale points).')


def save_landmarks(scene, P, tpl_data, scn_data):
    tpl_data, scn_data = np.asarray(tpl_data), np.asarray(scn_data)
    assert len(tpl_data) == len(scn_data) and tpl_data.shape[1:] == (3,), (
        f'need equal-length (N,3) point sets (ndim=3 on the Points layer), '
        f'got {tpl_data.shape} vs {scn_data.shape}')
    _check_points_in_bounds(P, tpl_data, scn_data)
    p = data_home + date + '_' + user + f'/series_{scene}/landmarks_s{scene}.pkl.gz'
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with gzip.open(p, 'wb') as f:
        pickle.dump({'template_points_idx_zyx': tpl_data, 'scene_points_idx': scn_data,
                     'rot180': bool(P.get('rot180', False))}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'{len(tpl_data)} landmark pairs (rot180={P.get("rot180", False)}) -> {p}')


def _check_rot180_matches(pts, P):
    if bool(pts.get('rot180', False)) != bool(P.get('rot180', False)):
        raise ValueError(
            f"landmarks were clicked with rot180={pts.get('rot180', False)} but this run has "
            f"rot180={P.get('rot180', False)} -- set ROT180 in cell [2] to match and re-run [2].")


def _fit_and_diagnose(scene, P, tpl_idx_zyx, scn_idx_zyx):
    template, moving = P['template'], P['moving']
    _check_points_in_bounds(P, tpl_idx_zyx, scn_idx_zyx)      # catch bad saved pickles on load
    tpl_idx = np.asarray(tpl_idx_zyx)[:, [2, 1, 0]]            # (Z,Y,X) display -> (X,Y,Z) for template
    scn_idx = np.asarray(scn_idx_zyx)                          # (Z,Y,X) matches moving
    tpl_phys = np.array([ants.transform_index_to_physical_point(template, [int(round(v)) for v in p]) for p in tpl_idx])
    scn_phys = np.array([ants.transform_index_to_physical_point(moving, [int(round(v)) for v in p]) for p in scn_idx])
    A, t = _fit_similarity_allow_reflection(scn_phys, tpl_phys)
    tf = ants.create_ants_transform(matrix=A, translation=t, dimension=3)
    path = f'/tmp/landmark_init_s{scene}.mat'
    ants.write_transform(tf, path)
    pred = np.array([tf.apply_to_point(tuple(p)) for p in tpl_phys])
    res = np.linalg.norm(pred - scn_phys, axis=1)
    sv = np.linalg.svd(A, compute_uv=False)
    print(f'  fit: det={np.linalg.det(A):+.2f} (neg = reflection, expected)  '
          f'scale_svd={np.round(sv, 2)}  residual mean={res.mean():.1f}um max={res.max():.1f}um')
    if res.max() > 20:
        print(f'  *** point {int(np.argmax(res))} residual {res.max():.1f}um — recheck that click/order')
    if np.any((sv < 0.3) | (sv > 3.0)):
        print('  *** scale far from 1 — transform will shrink/blow up the scene; re-click before trusting SyN')
    return path


def register_and_warp(scene, P, show=True):
    """Load saved landmarks -> similarity init -> SyN -> warp the MB mask onto the scene.
    NO trimming, NO save. Returns {reg, mb_reg} where mb_reg is the UNTRIMMED warped MB mask.
    QC plot: registration overlay, untrimmed MB mask on native nc82, warped nc82 projection."""
    lm = data_home + date + '_' + user + f'/series_{scene}/landmarks_s{scene}.pkl.gz'
    with gzip.open(lm, 'rb') as f:
        pts = pickle.load(f)
    _check_rot180_matches(pts, P)
    init = _fit_and_diagnose(scene, P, pts['template_points_idx_zyx'], pts['scene_points_idx'])

    reg = ants.registration(fixed=P['template'], moving=P['moving'], type_of_transform='SyN',
                            initial_transform=[init], verbose=False)
    mb_reg = ants.apply_transforms(fixed=P['moving'], moving=P['mb_mask_template'],
                                   transformlist=reg['invtransforms'],
                                   interpolator='nearestNeighbor').numpy() > 0.5
    print(f'  MB(reg, untrimmed) = {mb_reg.sum()} vox')

    if show:
        w, tn = reg['warpedmovout'].numpy(), P['template'].numpy()
        zc = int(np.argmax(w.sum(axis=(0, 1))))
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(tn[:, :, zc].T, cmap='Reds', alpha=.6)
        ax[0].imshow(w[:, :, zc].T, cmap='Greens', alpha=.6)
        ax[0].set_title(f's{scene} reg overlay (template=R, nc82=G, z={zc})')
        ax[1].imshow(P['nc82_stack'].max(0), cmap='gray')
        ax[1].imshow(mb_reg.max(0), cmap=ListedColormap(['none', 'blue']), alpha=.4)
        ax[1].set_title('nc82 max-proj + MB mask (UNTRIMMED)')
        ax[2].imshow(w.max(2).T, cmap='gray')
        ax[2].set_title('warped nc82 max-proj')
        for a in ax:
            a.axis('off')
        fig.tight_layout()
    return dict(reg=reg, mb_reg=mb_reg)


def to_raw(P, arr):
    """Map an array from the (possibly rot180) working frame back to the RAW nc82 stack
    orientation -- what the saved .tif looks like and what the other channels are in.
    Identity when rot180 is off."""
    return np.rot90(arr, 2, axes=(1, 2)) if P.get('rot180') else arr


def save_mask(scene, mask, suffix='', P=None):
    """Writes the mask to series_<s>/masks/. If P was prepped with rot180=True, the mask is
    rotated back to the RAW nc82 stack orientation first, so the saved .tif lines up with the
    original per-channel .tif files (not the rotated frame used for clicking)."""
    if P is not None:
        mask = to_raw(P, mask)
    d = data_home + date + '_' + user + f'/series_{scene}/masks/'
    os.makedirs(d, exist_ok=True)
    path = d + f'msk_MB_registered_s{scene}{suffix}.tif'
    imwrite(path, np.ascontiguousarray(mask).astype(np.uint8), imagej=True)
    print(f'saved {int(mask.sum())} vox{" (rotated back to raw orientation)" if P is not None and P.get("rot180") else ""} -> {path}')


def process_scene(scene, envelope='adaptive', suffix='', rot180=False, show=False):
    """Headless full run for batching: prep -> register + warp -> trim with the chosen
    envelope ('adaptive' | 'fixed' | None) at default params -> save. Set rot180 per scene
    (a dict {scene: True} works well) for upside-down brains."""
    P = prep_scene(scene, rot180=rot180)
    out = register_and_warp(scene, P, show=show)
    if envelope == 'adaptive':
        sig = defining_whole_mask_adaptative(P['nc82_stack'], P['vxy'], P['vz']).astype(bool)
    elif envelope == 'fixed':
        sig = defining_whole_mask(P['nc82_stack'], P['vxy'], P['vz']).astype(bool)
    else:
        sig = None
    mb_final = out['mb_reg'] & sig if sig is not None else out['mb_reg']
    save_mask(scene, mb_final, suffix, P=P)
    if sig is not None:
        save_mask(scene, sig, suffix + '_ALL', P=P)
    return dict(P=P, mb_final=mb_final, signal=sig, **out)

print('helpers ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════
scene  = 0        # <<<<<<<<<<  SELECT SCENE TO RUN  (0 .. len(scene_names)-1)
ROT180 = False    # True if this brain is imaged upside-down vs the template's top-down view:
                  # rotates nc82 180° in-plane for clicking + registration; the saved masks are
                  # rotated back to the raw orientation. Landmarks remember this setting.
# ══════════════════════════════════════════════════════════════════
P = prep_scene(scene, rot180=ROT180)
print(f'scene {scene}: {scene_names[scene]}   nc82 {P["nc82_stack"].shape}   '
      f'vxy={P["vxy"]:.4f} um  vz={P["vz"]:.4f} um' + ('   [ROT180]' if ROT180 else ''))

In [ ]:
# ─────────── HUMAN INPUT ───────────
# Click matching landmarks in BOTH viewers, SAME ORDER, spread across X/Y/Z (5–8 points).
# Use the "Add points" tool (press 2). Leave viewers open, then run the next cell.
vt, vs, tpl_layer, scn_layer = open_landmark_viewers(P)

In [ ]:
# run AFTER clicking. To drop a stray point first:
#   tpl_layer.data = np.delete(tpl_layer.data, i, 0)   /   scn_layer.data = np.delete(scn_layer.data, i, 0)
save_landmarks(scene, P, tpl_layer.data, scn_layer.data)

In [ ]:
out = register_and_warp(scene, P)   # loads landmarks_s{scene}, fit + SyN + warp; NO trim / NO save yet
# out['mb_reg'] = warped MB mask (UNTRIMMED) ;  out['reg'] = ANTs transforms

In [ ]:
# ─── signal-envelope tuning — re-run freely, it's fast: the blur is cached on P, only
#     changing SIGMA_UM re-blurs (~1 min); the percentiles are re-thresholded in <1 s ───
SIGMA_UM  = 0.8   # blur width; the ONLY slow knob
PCT       = 70    # fixed:    one global percentile on the blurred stack
BASE_PCT  = 70    # adaptive: percentile at/above the intensity-peak frame
STEP_PCT  = 5     # adaptive: percentile removed per STEP_DROP% drop in frame-mean past the peak
STEP_DROP = 10

blurred, frame_mean = get_blurred(P, SIGMA_UM)
mask_all       = signal_mask_fixed(blurred, PCT)
mask_all_adapt = signal_mask_adaptive(blurred, frame_mean, BASE_PCT, STEP_PCT, STEP_DROP)
print(f'fixed p{PCT}: {mask_all.sum()}   adaptive: {mask_all_adapt.sum()}   '
      f'| MB(reg) untrimmed: {out["mb_reg"].sum()}')

v = tune_viewer(scene, P, out['mb_reg'], mask_all, mask_all_adapt)   # reuses one window across re-runs

In [ ]:
# ─── choose the envelope, apply the trim, save BOTH the trimmed MB mask and the envelope ───
ENVELOPE = mask_all_adapt      # mask_all_adapt | mask_all | None   (None = save MB mask untrimmed)

mb_final  = out['mb_reg'] & ENVELOPE if ENVELOPE is not None else out['mb_reg']
all_final = ENVELOPE if ENVELOPE is not None else mask_all_adapt
print(f'MB(reg)={out["mb_reg"].sum()}  ->  trimmed={mb_final.sum()}  '
      f'({100 * mb_final.sum() / max(out["mb_reg"].sum(), 1):.0f}% kept)')

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, m, ttl, col in ((ax[0], out['mb_reg'], 'MB mask untrimmed', 'blue'),
                       (ax[1], mb_final, 'MB mask final (saved)', 'red')):
    a.imshow(P['nc82_stack'].max(0), cmap='gray')
    a.imshow(m.max(0), cmap=ListedColormap(['none', col]), alpha=.5)
    a.set_title(ttl); a.axis('off')
fig.tight_layout()

save_mask(scene, mb_final, P=P)                    # -> series_<s>/masks/msk_MB_registered_s<s>.tif
save_mask(scene, all_final, suffix='_ALL', P=P)    # -> series_<s>/masks/msk_MB_registered_s<s>_ALL.tif

In [ ]:
# ── frame-by-frame napari QC in the RAW orientation (what the saved .tif looks like,
#     and the frame the other channels are in) — to_raw() is identity when ROT180 is off ──
import napari
v = napari.Viewer(title=f'MB mask s{scene} — raw orientation')
v.add_image(to_raw(P, P['nc82_stack']), name='nc82 (raw)', colormap='gray')
v.add_labels(to_raw(P, mb_final).astype(np.uint8), name='MB mask (saved)', opacity=.5)

# ── optional: batch every scene that already has saved landmarks (default adaptive trim) ──
# ROT180_SCENES = {13}   # scenes imaged upside-down (must match how their landmarks were clicked)
# for s in range(len(scene_names)):
#     lp = data_home + date + '_' + user + f'/series_{s}/landmarks_s{s}.pkl.gz'
#     if not os.path.exists(lp):
#         print(f'scene {s}: no landmarks, skip'); continue
#     print(f'--- scene {s}: {scene_names[s]} ---')
#     process_scene(s, envelope='adaptive', rot180=s in ROT180_SCENES)